### Imports

In [2]:
import json
import numpy as np

from tqdm import tqdm
from sentence_transformers import SentenceTransformer

import faiss
import os

### Load chunks

In [4]:
# Load semantic chunks from previous step
INPUT_PATH = "../data/chunks/chunks.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 5932 chunks


### Initialize embedding model

In [6]:
# Lightweight and fast model (good default)
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Generate embeddings (batch mode)

In [ ]:
# Extract all texts
texts = [chunk["text"] for chunk in chunks]

# Encode in batch (much faster than loop)
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/186 [00:00<?, ?it/s]

### Create FAISS index

In [ ]:
# FAISS requires float32
vectors = embeddings.astype("float32")

# Create index (L2 distance)
index = faiss.IndexFlatL2(vectors.shape[1])

# Add vectors to index
index.add(vectors)

print("Total vectors in index:", index.ntotal)

### Save index

In [ ]:
INDEX_PATH = "../data/embeddings/faiss.index"
os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)

faiss.write_index(index, INDEX_PATH)

print("Index saved")

### Save metadata (IMPORTANT)

In [ ]:
# Save chunks separately (without embeddings)
META_PATH = "../data/embeddings/metadata.json"

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Metadata saved")

### Search function (test retrieval)

In [ ]:
def search(query, k=5):
    """
    Performs semantic search:
    - encodes query
    - finds nearest vectors
    - returns top-k chunks
    """
    # Encode query
    q_emb = model.encode([query]).astype("float32")
    
    # Search in FAISS
    distances, indices = index.search(q_emb, k)
    
    results = []
    
    for i in indices[0]:
        results.append(chunks[i])
    
    return results

In [ ]:
query = "churn prediction model CatBoost used"
results = search(query)

for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(r["text"][:300])

### Summary

- Loaded semantic chunks from previous step  
- Converted text into **vector embeddings** using a pre-trained model  
- Built a **FAISS index** for efficient similarity search  
- Stored vectors and metadata separately  
- Implemented a basic **semantic search function**  

👉 Result: working vector search system for retrieving relevant chunks in a RAG pipeline